# Lecture 15: Diagnostics And Transformations

This notebook uses housing data to inspect model assumptions and compare an outcome transformation.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan

sns.set_theme(style="whitegrid")

from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("Could not find repository root")


ROOT = find_repo_root()
DATA = ROOT / "data" / "raw"


In [ ]:
housing = pd.read_csv(DATA / "housing_sales.csv")
housing.head()


In [ ]:
linear = smf.ols(
    "price_k_eur ~ size_sq_m + rooms + age_years + renovation_score + near_transit + C(district)",
    data=housing,
).fit()
print(linear.summary())


In [ ]:
diagnostics = housing.assign(fitted=linear.fittedvalues, residual=linear.resid)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.scatterplot(data=diagnostics, x="fitted", y="residual", alpha=0.65, ax=axes[0])
axes[0].axhline(0, color="black", linewidth=1)
sns.histplot(diagnostics["residual"], kde=True, ax=axes[1])
axes[0].set(title="Residuals versus fitted")
axes[1].set(title="Residual distribution")


In [ ]:
bp_stat, bp_pvalue, _, _ = het_breuschpagan(linear.resid, linear.model.exog)
print(f"Breusch-Pagan statistic: {bp_stat:.2f}")
print(f"p-value: {bp_pvalue:.4f}")


In [ ]:
log_model = smf.ols(
    "np.log(price_k_eur) ~ size_sq_m + rooms + age_years + renovation_score + near_transit + C(district)",
    data=housing,
).fit()

pd.DataFrame(
    {
        "model": ["linear_price", "log_price"],
        "adjusted_r_squared": [linear.rsquared_adj, log_model.rsquared_adj],
        "aic": [linear.aic, log_model.aic],
    }
)


In [ ]:
influence = linear.get_influence()
housing.assign(cooks_distance=influence.cooks_distance[0]).nlargest(5, "cooks_distance")


## LLM Check

Ask an LLM what pattern it sees in the residual plot. Accept the answer only if it links the pattern to a specific model change you can fit and compare.
